# AI_lab_2 — Kaggle / Colab
Увімкніть GPU та Internet і під'єднайте competition dataset. Перша клітинка завантажить код із https://github.com/yelgor/AI_lab_2; архів не потрібен.

Для двох незалежних GPU-середовищ виберіть різні моделі або folds, використовуючи ті самі manifests. На одному невеликому GPU запускайте одне навчання.


In [ ]:
from pathlib import Path
import os, sys, subprocess

print(f'Python {sys.version.split()[0]}', flush=True)
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError('Потрібен Python 3.11, 3.12 або 3.13; перевірте версію середовища виконання.')

def bootstrap_run(arguments):
    # Forward pip/git output into the notebook instead of showing only
    # subprocess.CalledProcessError when an installation fails.
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Команда завершилася з кодом {returncode}. Причина наведена вище у виводі pip/git.')

PROJECT_DIR = Path('/kaggle/working/AI_lab_2' if Path('/kaggle').exists() else '/content/AI_lab_2')
if not PROJECT_DIR.exists():
    bootstrap_run(['git', 'clone', 'https://github.com/yelgor/AI_lab_2.git', str(PROJECT_DIR)])
elif (PROJECT_DIR / '.git').is_dir():
    bootstrap_run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'])
assert (PROJECT_DIR / 'pyproject.toml').is_file(), f'У {PROJECT_DIR} немає pyproject.toml; перевірте каталог'
os.chdir(PROJECT_DIR)
extras = '.' if Path('/kaggle').exists() else '.[kaggle]'
bootstrap_run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-e', extras])


Після першого встановлення залежностей перезапустіть Python-сеанс Colab (Runtime → Restart session / Среда выполнения → Перезапустить сеанс), щоб kernel підхопив установлені версії. Потім продовжуйте з наступної клітинки `doctor`. Internet потрібен для встановлення пакетів і першого завантаження pretrained weights. Підтримуються Python 3.11–3.13.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
PROJECT_DIR = Path('/kaggle/working/AI_lab_2' if Path('/kaggle').exists() else '/content/AI_lab_2')
os.chdir(PROJECT_DIR)
def run(*args):
    subprocess.run([sys.executable, '-m', 'cvlab', *map(str, args)], check=True)
run('doctor')


### Дані змагання в Colab

Змагання: https://www.kaggle.com/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026.

1. Увійдіть у свій Kaggle-акаунт та прийміть правила на сторінці змагання (Join Competition / Rules).
2. Відкрийте https://www.kaggle.com/settings/api → Generate New Token. Скопіюйте API token.
3. Запустіть наступну клітинку. Вставте токен у приховане поле вводу Colab. Або збережіть його в Colab Secrets з назвою `KAGGLE_API_TOKEN` і надайте notebook доступ.

Токен не вставляється в код, не друкується й не записується до репозиторію. Налаштовані раніше credentials Kaggle також підтримуються. На Kaggle ця клітинка пропускається: під'єднайте competition через Add Input.

Завантажуються приблизно 11 GB даних. Після завантаження ZIP розпаковується на локальний диск Colab; успішно розпакований архів видаляється, щоб звільнити місце. Повторне виконання використовує готові дані. HTTP 401 означає проблему авторизації; при HTTP 403 перевірте участь і прийняття правил під тим самим акаунтом. Реальне завантаження потребує вашого доступу до змагання.


In [ ]:
from pathlib import Path
import os, sys, subprocess
from cvlab.kaggle_data import ready_data_root

IS_KAGGLE = Path('/kaggle').exists()
DOWNLOAD_DIR = Path('/content/drone-data')
if IS_KAGGLE:
    print('Kaggle: використовуйте дані, під’єднані через Add Input.')
elif ready_data_root(DOWNLOAD_DIR) is not None:
    print('Дані вже завантажено й розпаковано.')
else:
    credential_dir = Path(os.environ.get('KAGGLE_CONFIG_DIR', str(Path.home() / '.kaggle')))
    has_credentials = bool(os.environ.get('KAGGLE_API_TOKEN')) or bool(
        os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')) or any(
        (credential_dir / name).is_file() for name in ('access_token', 'kaggle.json'))
    if not has_credentials:
        _api_token = None
        try:
            from google.colab import userdata
            _api_token = userdata.get('KAGGLE_API_TOKEN')
        except Exception:
            pass  # Missing/inaccessible optional Secret: use the hidden prompt below.
        if not _api_token:
            from getpass import getpass
            _api_token = getpass('Вставте Kaggle API token (ввід прихований): ')
        if not _api_token or not _api_token.strip():
            raise ValueError('Потрібен API token з https://www.kaggle.com/settings/api')
        os.environ['KAGGLE_API_TOKEN'] = _api_token.strip()
        del _api_token
    subprocess.run([sys.executable, '-m', 'cvlab.kaggle_data', '--out', str(DOWNLOAD_DIR)], check=True)


In [ ]:
from cvlab.kaggle_data import find_data_root
# Корінь має містити train/, val/, test_images.csv і sample_submission.csv.
if Path('/kaggle').exists():
    DATA_ROOT = find_data_root('/kaggle/input')
else:
    DATA_ROOT = find_data_root('/content/drone-data')
print('DATA_ROOT:', DATA_ROOT)
os.environ['CVLAB_DATA_ROOT'] = str(DATA_ROOT)
run('prepare', '--root', DATA_ROOT, '--out', 'prepared', '--folds', '3', '--workers', '8')


Перегляньте розподіли та GT перед навчанням. Під час паралельної роботи використовуйте однаковий каталог `prepared/` в обох середовищах.

In [ ]:
from IPython.display import display, Image
print(json.dumps(json.loads(Path('prepared/eda_train/summary.json').read_text()), indent=2)[:5000])
display(Image(filename='prepared/eda_train/distributions.png'))
for path in sorted(Path('prepared/eda_train').glob('gt_*.jpg'))[:2]:
    display(Image(filename=str(path)))


In [ ]:
import yaml
MODEL = 'fcos'  # fasterrcnn / retinanet / fcos / rtdetr; variants див. configs/
FOLD = 0       # 0, 1, 2; кожний запускається незалежно
SMOKE = True   # Спочатку коротка перевірка. False — повний run.
config = yaml.safe_load(Path(f'configs/{MODEL}.yaml').read_text())
config['data'].update(root=str(DATA_ROOT), prepared='prepared', fold=FOLD)
name = f"{MODEL}_fold{FOLD}" + ('_smoke' if SMOKE else '')
config['output'] = f'runs/{name}'
if SMOKE:
    config['train'].update(epochs=1, max_batches_per_epoch=2)
# Якщо OOM: batch_size=1. Для порівняння моделей узгодьте параметри.
# config['loader']['batch_size'] = 1
config_path = Path('run_configs') / f'{name}.yaml'
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path, config['output'])


Наступна клітинка запускає GPU-навчання. Smoke обмежує лише training batches; validation залишається повною. Повний run за замовчуванням має м’який ліміт 2 години і максимум 20 епох. Для повторного запуску задайте нове ім’я output або відновлюйте весь попередній run із `last.pt`.

In [ ]:
RESUME = False
arguments = ['train', '--config', config_path]
if RESUME:
    arguments += ['--resume', str(Path(config['output']) / 'last.pt')]
run(*arguments)


In [ ]:
import matplotlib.pyplot as plt
history = [json.loads(line) for line in (Path(config['output']) / 'history.jsonl').read_text().splitlines()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([r['elapsed_seconds']/3600 for r in history], [r['metrics']['coco100']['mAP'] for r in history], marker='o')
axes[0].set(xlabel='Elapsed hours', ylabel='COCO100 mAP', title=name)
axes[1].plot([r['epoch'] for r in history], [r['train_loss'] for r in history], marker='o')
axes[1].set(xlabel='Epoch', ylabel='Training loss')
plt.show()
print(json.dumps(json.loads((Path(config['output']) / 'best_metrics.json').read_text()), indent=2))


Для benchmark порівнюйте моделі на одному GPU, з однаковою precision, роздільністю та batch size. Batch latency не дорівнює затримці окремого кадру в онлайн-черзі.

In [ ]:
run('benchmark', '--config', config_path,
    '--checkpoint', Path(config['output']) / 'best.pt',
    '--set', 'loader.batch_size=1',
    '--out', Path(config['output']) / 'benchmark_batch1.json')


Submission генеруйте після повного навчання та вибору моделі. Ця клітинка не відправляє файл у змагання. Не використовуйте smoke checkpoint як фінальне рішення.

In [ ]:
GENERATE_SUBMISSION = False
if GENERATE_SUBMISSION:
    assert not SMOKE, 'Спершу виконайте повне навчання'
    run('predict', '--config', config_path,
        '--checkpoint', Path(config['output']) / 'best.pt',
        '--out', Path(config['output']) / 'submission.csv')


Збережіть весь каталог `runs/<name>/`, `prepared/` і `run_configs/` до завершення сесії. Для продовження відновіть їх за тими самими відносними шляхами. Реальні результати, failure cases і власні висновки ще потрібно отримати та проаналізувати; notebook не замінює авторський звіт.